# 🌑 LunarSight — Notebook 03: Polarimetric Analysis

**Agent 3**: Compute Stokes parameters, CPR, m-χ decomposition,
and diagnostic ice/rock flags from despeckled covariance data.

---

In [ ]:
# === Setup ===
import os
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/Lunar-Sight'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/YOUR_USERNAME/Lunar-Sight.git {REPO_DIR}
os.chdir(os.path.join(REPO_DIR, 'Lunar-Sight'))
!pip install -q -r requirements_colab.txt

In [ ]:
# === Run Agent 3 ===
import yaml, logging
logging.basicConfig(level=logging.INFO)

from agent3_polarimetry.agent import agent3_node

state = {
    'mission_config_path': 'config/mission_config.yaml',
    'despeckled_tensor_path': 'outputs/agent2/despeckled_tensor.npy',
    # Fallback: use raw tensor if agent 2 not run
    'raw_tensor_path': 'outputs/agent1/co_registered_tensor.npy',
}

result = agent3_node(state)
print(f"Status: {result.get('agent3_status')}")
print(f"Feature tensor: {result.get('polarimetric_tensor_path')}")

In [ ]:
# === Visualize CPR Map ===
import numpy as np
import matplotlib.pyplot as plt

cpr = np.load(result.get('cpr_l_path', 'outputs/agent3/l_cpr.npy'))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# CPR map
im0 = axes[0].imshow(cpr, cmap='RdYlBu_r', vmin=0, vmax=3)
axes[0].set_title('L-band CPR')
plt.colorbar(im0, ax=axes[0], label='CPR')
axes[0].contour(cpr, levels=[1.0], colors='red', linewidths=1.5)

# CPR histogram
valid = cpr[np.isfinite(cpr)]
axes[1].hist(valid.ravel(), bins=100, color='steelblue', alpha=0.7)
axes[1].axvline(x=1.0, color='red', linestyle='--', label='CPR=1 threshold')
axes[1].set_xlabel('CPR')
axes[1].set_ylabel('Count')
axes[1].set_title(f'CPR Distribution (n={len(valid):,})')
axes[1].legend()

pct_above_1 = np.sum(valid > 1) / len(valid) * 100
plt.suptitle(f'CPR > 1: {pct_above_1:.1f}% of pixels (ice candidates)')
plt.tight_layout()
plt.show()

In [ ]:
# === Visualize m-χ RGB ===
feat = np.load(result.get('polarimetric_tensor_path', 'outputs/agent3/polarimetric_feature_tensor.npy'))

# Channels: L_CPR, DOP, R, G, B, ...
# Find R/G/B channels
print(f'Feature tensor shape: {feat.shape}')

# If we have enough channels, plot m-chi RGB
if feat.shape[0] >= 6:
    r_idx, g_idx, b_idx = 3, 4, 5  # R_dbl, G_vol, B_srf
    R = feat[r_idx]
    G = feat[g_idx]
    B = feat[b_idx]
    
    # Normalize each channel to [0, 1] for display
    def norm(x):
        v = x[np.isfinite(x)]
        p5, p95 = np.percentile(v, [5, 95])
        return np.clip((x - p5) / max(p95 - p5, 1e-10), 0, 1)
    
    rgb = np.stack([norm(R), norm(G), norm(B)], axis=-1)
    rgb = np.nan_to_num(rgb)
    
    plt.figure(figsize=(10, 10))
    plt.imshow(rgb)
    plt.title('m-χ Decomposition RGB\nRed=Double-bounce | Green=Volume | Blue=Surface')
    plt.axis('off')
    plt.show()